# Foundation Models: Scaling Laws (Chinchilla) & Distributed Training

Pretraining modern Large Language Models requires balancing two fundamental pillars:
1. **Statistical Scaling Laws**: Determining the optimal balance between model parameters ($N$) and training tokens ($D$) for a fixed compute budget ($C$).
2. **Distributed Systems Parallelism**: Overcoming GPU memory and compute limits using Data Parallelism (DDP/FSDP), Tensor Parallelism (Megatron-LM), and Pipeline Parallelism (GPipe / 1F1B).

| Strategy | What is Split | Key Framework | Interconnect Requirement |
|---|---|---|---|
| **Data Parallel (DDP / FSDP)** | Batch dimension / Model Sharding | PyTorch FSDP, DeepSpeed ZeRO | Standard Ethernet / PCIe |
| **Tensor Parallel (TP)** | Intra-layer weight matrices ($W_{col}, W_{row}$) | Megatron-LM | Ultra-high speed (NVLink 900 GB/s) |
| **Pipeline Parallel (PP)** | Inter-layer stages | DeepSpeed Pipeline, Megatron | Low-to-moderate bandwidth |
| **3D Parallelism** | All three dimensions combined | Megatron-DeepSpeed | Hierarchical NVLink + InfiniBand |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
print(f'MPS available    : {torch.backends.mps.is_available()}')
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Active device    : {device}')


---
## 1. LLM Pretraining Objective & Learning Dynamics

Autoregressive language models are trained via **causal next-token prediction** using cross-entropy loss:

$$\mathcal{L}_{\text{LM}} = -\frac{1}{T} \sum_{t=1}^{T} \log P(x_t \mid x_{<t}; \theta)$$

Perplexity measures the effective branching factor: $\text{PPL} = \exp(\mathcal{L})$. Lower perplexity indicates better predictive uncertainty.

Standard pretraining optimizer recipes rely on **AdamW** with a **warmup + cosine decay** learning rate schedule:
- **Linear Warmup**: 2,000–5,000 steps to stabilize gradient variance.
- **Cosine Decay**: Gradually reduces learning rate down to 10% of maximum ($0.1 \cdot \eta_{\max}$).
- **Weight Decay**: 0.1 to regularize parameters without penalizing LayerNorm/RMSNorm or biases.


In [ ]:
# Autoregressive Next-Token Loss & Cosine Schedule Simulation
import math

def compute_autoregressive_loss(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, float]:
    """
    logits : (B, T, V)  - unnormalized class scores
    targets: (B, T)     - token IDs shifted right
    Returns: cross-entropy loss scalar and perplexity
    """
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.reshape(B * T, V), targets.reshape(B * T))
    perplexity = math.exp(min(loss.item(), 20.0))  # guard against overflow
    return loss, perplexity

def get_cosine_lr(step: int, warmup_steps: int, max_steps: int, lr_max: float, lr_min: float) -> float:
    if step < warmup_steps:
        return lr_max * (step / max(1, warmup_steps))
    if step > max_steps:
        return lr_min
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return lr_min + coeff * (lr_max - lr_min)

# Visualize Cosine Schedule
steps = np.arange(10000)
lrs = [get_cosine_lr(s, warmup_steps=1000, max_steps=10000, lr_max=3e-4, lr_min=3e-5) for s in steps]

plt.figure(figsize=(8, 3.5))
plt.plot(steps, lrs, color='steelblue', lw=2)
plt.axvline(1000, color='crimson', ls='--', label='Warmup End (1k steps)')
plt.title('Pretraining Learning Rate Schedule (Warmup + Cosine Decay)')
plt.xlabel('Training Steps')
plt.ylabel('Learning Rate')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


---
## 2. Scaling Laws: Kaplan vs. Chinchilla (Hoffmann et al., 2022)

### Compute Budget Formula
During Transformer pretraining, total floating point operations (FLOPs) $C$ are dominated by matrix multiplications:

$$C \approx 6 \times N \times D$$

- **Forward pass**: $\approx 2N$ FLOPs per token (1 multiply-accumulate = 2 FLOPs).
- **Backward pass**: $\approx 4N$ FLOPs per token (computing input gradients and weight gradients).

### Kaplan et al. (OpenAI, 2020) vs. Chinchilla (DeepMind, 2022)

| Aspect | Kaplan et al. (2020) | Chinchilla (Hoffmann et al., 2022) |
|:---|:---|:---|
| **Parameter Scaling** | $N \propto C^{0.73}$ (scale parameters fast) | $N \propto C^{0.50}$ (scale parameters equally) |
| **Token Scaling** | $D \propto C^{0.27}$ (scale tokens slowly) | $D \propto C^{0.50}$ (scale tokens equally) |
| **Optimal Token/Param Ratio** | $\approx 1.7$ tokens per parameter | **$\approx 20$ tokens per parameter** |
| **Historical Implication** | GPT-3 (175B on 300B tokens) was **severely undertrained** | Chinchilla (70B on 1.4T tokens) outperformed GPT-3 with identical compute |

### Modern "Inference-Optimal" Overtraining
While Chinchilla optimizes **training compute efficiency**, production serving is dominated by **inference costs**. Modern open models deliberately overtrain far beyond Chinchilla optimality:
- **LLaMA-1 / LLaMA-2 7B**: 1T–2T tokens (150–300 tokens/param)
- **LLaMA-3 / 3.2 8B**: 15T tokens (**1,875 tokens/param**)
- **Gemma 2 9B**: 8T tokens (**888 tokens/param**)


In [ ]:
# Chinchilla Loss Formulation & Optimal Frontier
# Loss L(N, D) = E + A / N^alpha + B / D^beta (Hoffmann et al., 2022)
E = 1.69
A = 406.4
B = 410.7
alpha = 0.34
beta = 0.28

def chinchilla_loss(N_billions, D_billions):
    N = N_billions * 1e9
    D = D_billions * 1e9
    return E + (A / (N ** alpha)) + (B / (D ** beta))

def chinchilla_optimal_params_and_tokens(C_flops: float):
    """Given compute budget C (FLOPs), compute optimal N (parameters) and D (tokens)."""
    # Under alpha * G = beta * H where C = 6 * N * D
    # Approximated Chinchilla rule: N ~ 0.05 * sqrt(C), D ~ sqrt(C) / (6 * 0.05) ~ 3.33 * sqrt(C)
    # Yields D ~ 20 * N
    N_opt = math.sqrt(C_flops / 6.0 / 20.0)
    D_opt = 20.0 * N_opt
    return N_opt, D_opt

# Compute table for standard budgets
budgets = [
    ('GPT-3 Budget (3.14e23 FLOPs)', 3.14e23),
    ('Chinchilla Budget (5.76e23 FLOPs)', 5.76e23),
    ('LLaMA-3 8B Compute (7.2e23 FLOPs)', 7.2e23),
]

print(f"{'Model / Budget':<35} | {'Optimal Params (N*)':<20} | {'Optimal Tokens (D*)':<20} | {'Actual Model Ratio'}")
print('-' * 95)
for name, c_fl in budgets:
    n_star, d_star = chinchilla_optimal_params_and_tokens(c_fl)
    print(f"{name:<35} | {n_star/1e9:>6.1f} Billion params  | {d_star/1e12:>6.2f} Trillion tokens | D* / N* = 20.0")

# Plotting Chinchilla IsoFLOP Contours
N_grid = np.logspace(0, 2.5, 100)  # 1B to ~300B parameters
D_grid = np.logspace(0, 3.5, 100)  # 1B to ~3000B tokens
NN, DD = np.meshgrid(N_grid, D_grid)
LL = chinchilla_loss(NN, DD)

plt.figure(figsize=(9, 5))
cs = plt.contour(NN, DD, LL, levels=25, cmap='viridis')
plt.clabel(cs, inline=True, fontsize=8, fmt='%.2f')

# Chinchilla optimal line (D = 20 * N)
plt.plot(N_grid, 20 * N_grid, color='red', lw=2.5, ls='--', label='Chinchilla Optimal Line (D = 20N)')
# Scatter notable models
plt.scatter([175], [300], color='crimson', s=100, zorder=5, label='GPT-3 175B (Undertrained: 1.7 tok/param)')
plt.scatter([70], [1400], color='gold', s=100, zorder=5, label='Chinchilla 70B (Optimal: 20 tok/param)')
plt.scatter([8], [15000], color='deepskyblue', s=100, zorder=5, label='LLaMA-3 8B (Inference-Overtrained: 1875 tok/param)')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Parameters N (Billions)')
plt.ylabel('Tokens D (Billions)')
plt.title('Chinchilla Loss Landscape & Compute-Optimal Frontier')
plt.legend(loc='lower right', fontsize=8)
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. Data Parallelism (DDP) & Gradient All-Reduce

Each GPU holds a **full copy of the model parameters** but processes a distinct shard of the global batch.
At the backward pass, gradients from each rank are **all-reduced** (averaged) across all GPUs using Ring-AllReduce or Tree-AllReduce via NCCL.


In [ ]:
# DDP gradient averaging simulation
torch.manual_seed(42)
world_size = 4
d_model = 64

# Each GPU starts with identical model weights
shared_weights = torch.randn(d_model, d_model)
models = [shared_weights.clone().requires_grad_(True) for _ in range(world_size)]

# Simulate different mini-batches on each GPU
def fake_loss(w, rank_seed):
    torch.manual_seed(rank_seed)
    x = torch.randn(8, d_model)
    return (x @ w).pow(2).mean()

# Each GPU computes its local gradient
grads = []
for rank, model_w in enumerate(models):
    loss = fake_loss(model_w, rank_seed=rank)
    loss.backward()
    grads.append(model_w.grad.clone())

# All-Reduce (average) across ranks
avg_grad = torch.stack(grads).mean(dim=0)

print('Gradient norms across ranks:')
for rank, g in enumerate(grads):
    print(f'  GPU {rank}: local_grad_norm={g.norm():.4f}')
print(f'  All-Reduced synchronized grad_norm: {avg_grad.norm():.4f} (applied by every GPU)')


---
## 4. ZeRO (Zero Redundancy Optimizer) & Fully Sharded Data Parallel (FSDP)

Standard DDP replicates everything on every GPU. For 16-bit mixed precision (FP16/BF16) with Adam:
- **Parameters ($P$)**: 2 bytes / param
- **Gradients ($G$)**: 2 bytes / param
- **Optimizer States ($O$)**: 16 bytes / param (FP32 master weight 4B + FP32 momentum 4B + FP32 variance 4B + FP32 grad 4B)
- **Total Static Model Memory**: **20 bytes per parameter** (before activations!)

For a **70 Billion parameter model**, static memory alone requires $70 \times 20\text{ GB} = \mathbf{1{,}400\text{ GB}}$! A single 80 GB A100 GPU cannot even store the weights, let alone train.

### ZeRO Partitioning Stages (Rajbhandari et al., DeepSpeed / PyTorch FSDP):
1. **ZeRO-1 ($P_{os}$)**: Shards **Optimizer States** across GPUs (memory reduction $\approx 4\times$).
2. **ZeRO-2 ($P_{os+g}$)**: Shards **Optimizer States + Gradients** across GPUs (memory reduction $\approx 8\times$).
3. **ZeRO-3 ($P_{os+g+p}$ / FSDP)**: Shards **Optimizer States + Gradients + Model Parameters**. Each GPU only holds $1/N_{\text{ranks}}$ of model weights, gathering parameters on-the-fly via All-Gather during forward/backward passes.


In [ ]:
# Calculate GPU memory requirements across ZeRO stages
from rich.table import Table
from rich.console import Console

console = Console()
table = Table(title='GPU Static Memory per Model (Gigabytes, FP16 + AdamW)', show_lines=True)
table.add_column('Model Size', style='cyan bold')
table.add_column('Standard DDP (1 GPU)', style='red')
table.add_column('ZeRO-1 (8 GPUs)', style='yellow')
table.add_column('ZeRO-2 (8 GPUs)', style='magenta')
table.add_column('ZeRO-3 / FSDP (8 GPUs)', style='green bold')

for model_name, params_b in [('7B (e.g. Mistral 7B)', 7), ('13B (e.g. LLaMA 13B)', 13), ('70B (e.g. LLaMA-3 70B)', 70)]:
    mem_ddp = params_b * 20.0
    mem_zero1 = (params_b * 4.0) + (params_b * 16.0 / 8.0)        # 2B params + 2B grads + 16B/8 opt
    mem_zero2 = (params_b * 2.0) + (params_b * 18.0 / 8.0)        # 2B params + (2B grad + 16B opt)/8
    mem_zero3 = (params_b * 20.0 / 8.0)                          # all 20B / 8
    table.add_row(
        model_name,
        f'{mem_ddp:.0f} GB',
        f'{mem_zero1:.1f} GB',
        f'{mem_zero2:.1f} GB',
        f'{mem_zero3:.1f} GB'
    )

console.print(table)


---
## 5. Tensor (Model) Parallelism: Megatron-LM MLP Block

In **Megatron-LM**, individual matrix multiplications are split across GPUs.
To minimize communication in a 2-layer MLP block ($y = \text{GELU}(x W_1) W_2$):
1. **Layer 1: ColumnParallelLinear**: Splits $W_1$ by columns: $W_1 = [W_{1,1} \mid W_{1,2}]$.
   - GPU $i$ computes $Z_i = \text{GELU}(x W_{1,i})$ with **no communication**!
2. **Layer 2: RowParallelLinear**: Splits $W_2$ by rows: $W_2 = \begin{bmatrix} W_{2,1} \\ W_{2,2} \end{bmatrix}$.
   - GPU $i$ computes partial product: $Y_i = Z_i W_{2,i}$.
3. **Single All-Reduce**: $Y = Y_1 + Y_2$ synchronizes output across GPUs!

**Key Insight**: By combining Column-Parallel with Row-Parallel, the non-linearity (GELU) executes locally on each GPU, requiring only **one communication step per MLP block**!


In [ ]:
class ColumnParallelLinear(nn.Module):
    """Splits out_features across world_size GPUs (column-wise partition)."""
    def __init__(self, in_features: int, out_features: int, world_size: int = 2):
        super().__init__()
        self.world_size = world_size
        self.out_per_rank = out_features // world_size
        self.shards = nn.ParameterList([
            nn.Parameter(torch.randn(in_features, self.out_per_rank) * 0.02)
            for _ in range(world_size)
        ])

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        # Each GPU computes its local column partition: (B, T, out_per_rank)
        return [x @ shard for shard in self.shards]

class RowParallelLinear(nn.Module):
    """Splits in_features across world_size GPUs (row-wise partition) + All-Reduce."""
    def __init__(self, in_features: int, out_features: int, world_size: int = 2):
        super().__init__()
        self.world_size = world_size
        self.in_per_rank = in_features // world_size
        self.shards = nn.ParameterList([
            nn.Parameter(torch.randn(self.in_per_rank, out_features) * 0.02)
            for _ in range(world_size)
        ])

    def forward(self, partial_inputs: list[torch.Tensor]) -> torch.Tensor:
        # Each GPU computes local partial sum, then All-Reduce (sums across ranks)
        partial_outputs = [z @ shard for z, shard in zip(partial_inputs, self.shards)]
        # All-Reduce sum
        return torch.stack(partial_outputs).sum(dim=0)

# Build Full Megatron-LM Tensor Parallel MLP Block
class TensorParallelMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int, world_size: int = 2):
        super().__init__()
        self.col_linear = ColumnParallelLinear(d_model, d_ff, world_size)
        self.row_linear = RowParallelLinear(d_ff, d_model, world_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Column Parallel (no comms)
        h_shards = self.col_linear(x)
        # 2. Local GELU activation on each rank
        act_shards = [F.gelu(h) for h in h_shards]
        # 3. Row Parallel + All-Reduce
        out = self.row_linear(act_shards)
        return out

tp_mlp = TensorParallelMLP(d_model=128, d_ff=512, world_size=4)
x_test = torch.randn(8, 32, 128)
y_tp = tp_mlp(x_test)

print(f'Input shape : {x_test.shape}')
print(f'Output shape: {y_tp.shape}')
print('Megatron-LM TP MLP successfully computed with 1 All-Reduce across 4 ranks!')


---
## 6. Pipeline Parallelism: GPipe & 1F1B Scheduling

Pipeline Parallelism partitions layers across $P$ pipeline stages (GPUs).
To prevent GPUs from idling (the **pipeline bubble**), the batch is split into $M$ **micro-batches**.

In the **1F1B (One Forward, One Backward)** schedule (DeepSpeed / Megatron-LM):
- Each stage alternates between computing one forward micro-batch and one backward micro-batch.
- This bounds the peak activation memory to at most $P$ in-flight micro-batches.

$$\text{Bubble Fraction} = \frac{P - 1}{M + P - 1}$$

For $P=4$ pipeline stages and $M=16$ micro-batches, the bubble overhead is only $\frac{3}{19} \approx 15.8\%$.


In [ ]:
# Visualize Pipeline Parallelism 1F1B Schedule
P = 4   # 4 Pipeline stages (GPUs)
M = 8   # 8 Micro-batches

bubble_ratio = (P - 1) / (M + P - 1)
print(f'Pipeline Stages (P)   : {P}')
print(f'Micro-batches (M)     : {M}')
print(f'Theoretical Bubble    : {bubble_ratio * 100:.1f}% idle time')

# Build Gantt Schedule Plot
fig, ax = plt.subplots(figsize=(11, 4.5))
colors_fwd = plt.cm.Blues(np.linspace(0.4, 0.9, M))
colors_bwd = plt.cm.Oranges(np.linspace(0.4, 0.9, M))

# Forward schedule: stage p starts micro-batch m at time p + m
for p in range(P):
    for m in range(M):
        t_start = p + m
        ax.barh(p, 1, left=t_start, color=colors_fwd[m], edgecolor='black', alpha=0.9)
        ax.text(t_start + 0.5, p, f'F{m}', ha='center', va='center', fontsize=7, color='white', weight='bold')

# Backward schedule
for p in range(P):
    for m in range(M):
        # In 1F1B, backward starts after stage P-1 finishes its first forward pass
        t_start = (P - 1 + M) + (P - 1 - p) + m
        ax.barh(p, 1, left=t_start, color=colors_bwd[m], edgecolor='black', alpha=0.9)
        ax.text(t_start + 0.5, p, f'B{m}', ha='center', va='center', fontsize=7, color='white', weight='bold')

ax.set_yticks(range(P))
ax.set_yticklabels([f'Stage {p} (GPU {p})' for p in range(P)])
ax.set_xlabel('Clock Cycles / Time Steps')
ax.set_title(f'Pipeline Parallelism 1F1B Schedule (Stages P={P}, Microbatches M={M}, Bubble={bubble_ratio*100:.1f}%)')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


---
## Summary & Key Takeaways

1. **Chinchilla Optimal Scaling**: Model parameters $N$ and tokens $D$ should scale in equal proportions ($N, D \propto C^{0.5}$), targeting $\approx 20$ tokens per parameter for compute-optimal training.
2. **Inference Overtraining**: Production open models (LLaMA-3, Gemma, Mistral) train on 15T+ tokens (100–1,800 tokens/param) to maximize inference throughput and lower per-token serving costs.
3. **ZeRO / FSDP Memory Sharding**: Eliminates memory redundancy by partitioning optimizer states (ZeRO-1), gradients (ZeRO-2), and parameters (ZeRO-3) across GPUs, reducing per-GPU memory from 20 bytes/param down to $20/N_{\text{ranks}}$ bytes/param.
4. **Tensor Parallelism (TP)**: Megatron-LM pairs Column-Parallel and Row-Parallel linear layers to keep non-linear activations local, requiring only a single All-Reduce synchronization per MLP block.
5. **Pipeline Parallelism (PP)**: Uses 1F1B micro-batch scheduling to bound activation memory and compress the pipeline bubble to $(P-1)/(M+P-1)$.
